In [ ]:
#------------------------------------------------ Import Lib ----------------------------------------
import re
import os
import datetime
import requests
import pandas as pd
import pdfplumber

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


In [ ]:
#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'KE CBK' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now = datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])

try:
    scriptfolder = os.path.dirname(os.path.abspath(__file__)) ## production environment (.py)
except NameError:
    scriptfolder = os.getcwd() ## notebook environment

os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


In [ ]:
#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': []}

regdict={

        regulatorName+' 1': 'https://www.centralbank.go.ke/wp-content/uploads/2023/06/Directory-of-Licenced-Commercial-Banks-Authorised-NOHCs-June-2023.pdf',
        regulatorName+' 2': 'https://www.centralbank.go.ke/wp-content/uploads/2026/06/Directory-of-Licenced-Money-Remittance-Providers-June-2026.pdf',
        regulatorName+' 3': 'https://www.centralbank.go.ke/wp-content/uploads/2026/02/Directory-of-Licenced-Microfinance-Banks-Feb-2026.pdf',
        regulatorName+' 4': 'https://www.centralbank.go.ke/wp-content/uploads/2026/06/Directory-of-Licenced-Foreign-Exchange-Bureaus-June-2026.pdf',
        regulatorName+' 5': 'https://www.centralbank.go.ke/wp-content/uploads/2022/11/Directory-of-Licensed-CRBs-November-2022.pdf',
        regulatorName+' 6': 'https://www.centralbank.go.ke/wp-content/uploads/2025/07/Directory-of-Authorised-Representative-Offices-July-2025.pdf',
        regulatorName+' 7': 'https://www.centralbank.go.ke/wp-content/uploads/2026/04/Directory-of-Digital-Credit-Providers-April-2026.pdf',
        }

Typology={

       regulatorName + ' 1': 'Commercial Banks & Mortgage Finance',
       regulatorName + ' 2': 'Licensed Money Remittance Providers',
       regulatorName + ' 3': 'List of licensed Microfinance Banks',
       regulatorName + ' 4': 'Licensed Forex Bureau',
       regulatorName + ' 5': 'Licensed Credit Reference Bureaus',
       regulatorName + ' 6': 'Representative Offices of Foreign Banks in Kenya',
       regulatorName + ' 7': 'Digital Credit Providers',

        }

# ListLabel rule (per ticket owner): 1 = bank named in list name, 2 = insurance,
# 3 = bank & insurance, 4 = other
ListLabeldict={

       regulatorName + ' 1': 1,   # Commercial Banks & Mortgage Finance
       regulatorName + ' 2': 4,   # Licensed Money Remittance Providers
       regulatorName + ' 3': 1,   # List of licensed Microfinance Banks
       regulatorName + ' 4': 4,   # Licensed Forex Bureau
       regulatorName + ' 5': 4,   # Licensed Credit Reference Bureaus
       regulatorName + ' 6': 1,   # Representative Offices of Foreign Banks in Kenya
       regulatorName + ' 7': 4,   # Digital Credit Providers

        }

processdate = now.strftime('%Y-%m-%d')

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36'}

# Lists whose PDF is a numbered "profile block" directory (labelled fields under each entity)
BLOB_LISTS = {regulatorName+' 1', regulatorName+' 3', regulatorName+' 5', regulatorName+' 6', regulatorName+' 7'}
# Lists whose PDF is a real 5-column table: No. | Name | Location | Contact Details | Date of Licencing
TABLE_LISTS = {regulatorName+' 2', regulatorName+' 4'}

In [ ]:
#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


# ---- page furniture (headers / footers / section titles) to drop, line by line ----
SKIP_PATTERNS = [
    r'^CENTRAL BANK OF KENYA$',
    r'^DIRECTORY OF ',
    r'^NO\. DIRECTORY OF',
    r'^INSTITUTIONS AND AUTHORISED',
    r'^COMPANIES$',
    r'^UPDATED ON ',
    r'^Page \d+ of \d+$',
    r'^C2: CBK - Official$',
    r'^\d{1,3}$',          # bare page-number footers
    r'^[A-C][:.]\s',       # section headers e.g. "A: COMMERCIAL BANKS"
]
SKIP_RES = [re.compile(p) for p in SKIP_PATTERNS]

# numbered entry start: "12." / "12. Name..." / "12 Name..." (list 5 has no dot)
ENTRY_RE = re.compile(r'^(\d{1,3})(\.)?(?:\s+|$)(.*)$')

# whitelisted field labels seen across the 7 CBK directories (longest first when compiled)
LABELS = ['Postal Address', 'Physical Address', 'Physical Location', 'P.O. Box', 'Address',
          'Telephone Contacts', 'Telephone Number', 'Telephone No.', 'Telephone No', 'Telephone',
          'Contact Centre Tel', 'Switch Board Tel', 'Contact Centre', 'Mobile No.', 'Mobile',
          'Sms', 'Tel', 'Fax',
          'E-mail address', 'Email address', 'Email Address', 'Official Email', 'E-mail', 'Email',
          'Website Address', 'Website',
          'Date Licenced', 'Date Licensed', 'Date Authorised', 'Date Authorized',
          'Peer Group', 'Branches', 'Sub-branches', 'Sales Centres', 'Sales Centre', 'Agencies',
          'Licenced Subsidiary', 'Chief Representative Officer', 'Interim Office Manager']
LABEL_RE = re.compile(r'(?<![A-Za-z-])(' + '|'.join(re.escape(l) for l in sorted(LABELS, key=len, reverse=True)) + r')\s*:\s*', re.I)

FIELD_MAP = {'postal address': 'postal', 'p.o. box': 'postal', 'address': 'postal',
             'physical address': 'physical', 'physical location': 'physical',
             'telephone contacts': 'phone', 'telephone number': 'phone', 'telephone no.': 'phone',
             'telephone no': 'phone', 'telephone': 'phone', 'tel': 'phone',
             'mobile no.': 'mobile', 'mobile': 'mobile',
             'fax': 'fax',
             'e-mail address': 'email', 'email address': 'email', 'official email': 'email',
             'e-mail': 'email', 'email': 'email',
             'website address': 'website', 'website': 'website',
             'date licenced': 'regdate', 'date licensed': 'regdate',
             'date authorised': 'regdate', 'date authorized': 'regdate'}

EMAIL_RE = re.compile(r"[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}", re.I)
WEB_RE = re.compile(r'(?:https?://[^\s,;]+|www\.[^\s,;]+)', re.I)
TEL_RE = re.compile(r'(?:Tel|Cell|Mobile|Phone)\s*[:.]?\s*([+\d][\d\s,/+()\-]*\d)', re.I)
FAX_RE = re.compile(r'Fax\s*[:.]?\s*([+\d][\d\s,/+()\-]*\d)', re.I)


def clean(s):
    return re.sub(r'\s+', ' ', s or '').strip()


def fix_split_first_letter(s):
    # PDF kerning artifact: "W akanda Credit" -> "Wakanda Credit", "S hujaa Mall" -> "Shujaa Mall"
    return re.sub(r'^([A-Z])\s+([a-z])', r'\1\2', s)


def fix_nairobi(s):
    # same artifact inside address cells: "N AIROBI" -> "NAIROBI"
    return re.sub(r'\bN\s+AIROBI\b', 'NAIROBI', s)


def extract_city(text):
    if not text:
        return ''
    s = clean(text).rstrip('.').strip()
    s = re.sub(r',?\s*Kenya$', '', s, flags=re.I).strip().rstrip(',').strip()
    parts = [p.strip() for p in s.split(',') if p.strip()]
    if parts:
        cand = parts[-1]
        if re.fullmatch(r"[A-Za-z][A-Za-z .’'\-]*", cand) and len(cand.split()) <= 2:
            return cand
    m = re.search(r"([A-Z][A-Za-z’'\-]{2,})$", s)
    return m.group(1) if m else ''


def extract_zip(text):
    if not text:
        return ''
    hits = re.findall(r'(?:[–—-]|,)\s*(\d{5})\b', text)
    return hits[-1] if hits else ''


def parse_blob_pdf(path, allow_restart=False):
    """Split a numbered profile-block directory PDF into one text blob per entity.
    allow_restart=True lets the numbering restart at '1.' (list 1 has sections A/B/C)."""
    lines = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            txt = page.extract_text() or ''
            for line in txt.split('\n'):
                line = line.strip()
                if not line or any(p.match(line) for p in SKIP_RES):
                    continue
                lines.append(line)
    entries, buf, expected = [], None, 1
    for line in lines:
        m = ENTRY_RE.match(line)
        if m:
            num, dot, rest = int(m.group(1)), m.group(2), m.group(3).strip()
            is_new = (num == expected) or (allow_restart and dot and num == 1)
            if is_new and (dot or rest):
                if buf is not None:
                    entries.append(' '.join(buf))
                buf = [rest] if rest else []
                expected = num + 1
                continue
        if buf is not None:
            buf.append(line)
    if buf is not None:
        entries.append(' '.join(buf))
    return entries


def parse_entry_fields(blob):
    """Name = text before the first whitelisted label; fields = slices between labels."""
    matches = list(LABEL_RE.finditer(blob))
    if not matches:
        return clean(blob), {}
    name = clean(blob[:matches[0].start()]).strip(' .,-')
    fields = {}
    for i, m in enumerate(matches):
        key = FIELD_MAP.get(m.group(1).lower())
        end = matches[i + 1].start() if i + 1 < len(matches) else len(blob)
        val = clean(blob[m.end():end]).strip(' ,;')
        if key and key not in fields and val:
            fields[key] = val
    return name, fields


def parse_contact_cell(contact):
    """Contact Details cell of lists 2/4 -> (postal, phone, fax, email, website)."""
    contact = fix_nairobi(clean(contact))
    emails = EMAIL_RE.findall(contact)
    email = '; '.join(emails)
    m = WEB_RE.search(EMAIL_RE.sub(' ', contact))
    website = m.group(0).strip('.,;)') if m else ''
    m = TEL_RE.search(contact)
    phone = clean(m.group(1)).strip(' ,;/') if m else ''
    m = FAX_RE.search(contact)
    fax = clean(m.group(1)).strip(' ,;/') if m else ''
    cut = len(contact)
    for r in (TEL_RE, FAX_RE, EMAIL_RE, WEB_RE):
        mm = r.search(contact)
        if mm:
            cut = min(cut, mm.start())
    postal = clean(contact[:cut]).strip(' ,;.-')
    postal = re.sub(r'(Email|E-mail|Tel|Fax|Website)\s*[:.]?\s*$', '', postal, flags=re.I).strip(' ,;.-')
    return postal, phone, fax, email, website


def append_row(reg, name, address_1, address_2, city, zip_, phone, fax, email, website, regdate):
    sqldict['Name'].append(name)
    sqldict['Address_1'].append(address_1)
    sqldict['Address_2'].append(address_2)
    sqldict['City'].append(city)
    sqldict['Zip'].append(zip_)
    sqldict['Phone'].append(phone)
    sqldict['Fax'].append(fax)
    sqldict['Email'].append(email)
    sqldict['Website'].append(website)
    sqldict['RegulationDate'].append(regdate)
    sqldict['RegulationType'].append('Regulated')
    sqldict['ListLabel'].append(ListLabeldict[reg])
    sqldict['ListName'].append(Typology[reg])
    sqldict['RegCtry'].append(reg.split()[0])
    sqldict['RegCode'].append(reg.split()[1])
    sqldict['ListCode'].append(reg.split()[2])
    sqldict['ListProcessDate'].append(processdate)

In [ ]:
#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):
    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")
    rows_before = len(sqldict['Name'])

    pdf_path = os.path.join(tempfolder, reg.replace(' ', '_') + '.pdf')
    resp = requests.get(regdict[reg], headers=HEADERS, verify=False, timeout=120)
    resp.raise_for_status()
    if not resp.content.startswith(b'%PDF'):
        raise ValueError(f'{reg}: downloaded file is not a PDF ({regdict[reg]})')
    with open(pdf_path, 'wb') as f:
        f.write(resp.content)

    if reg in BLOB_LISTS:
        entries = parse_blob_pdf(pdf_path, allow_restart=(reg == regulatorName + ' 1'))
        for blob in entries:
            name, fields = parse_entry_fields(blob)
            name = fix_split_first_letter(clean(name))
            if not name:
                continue
            postal = fix_nairobi(fields.get('postal', ''))
            physical = fix_nairobi(fields.get('physical', ''))
            phone = fields.get('phone', '') or fields.get('mobile', '')
            email = '; '.join(EMAIL_RE.findall(fields.get('email', '') or blob))
            wm = WEB_RE.search(fields.get('website', '')) or WEB_RE.search(EMAIL_RE.sub(' ', blob))
            website = wm.group(0).strip('.,;)') if wm else ''
            append_row(reg,
                       name,
                       physical or postal,
                       postal if physical else '',
                       extract_city(postal) or extract_city(physical),
                       extract_zip(postal),
                       phone,
                       fields.get('fax', ''),
                       email,
                       website,
                       fields.get('regdate', ''))

    elif reg in TABLE_LISTS:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                for tbl in page.extract_tables():
                    for row in tbl:
                        cells = [clean((c or '').replace('\n', ' ')) for c in row]
                        # keep only real data rows: col 0 is the running number
                        if len(cells) < 5 or not re.fullmatch(r'\d{1,3}\.?', cells[0]):
                            continue
                        name = fix_split_first_letter(cells[1])
                        if not name:
                            continue
                        location = fix_nairobi(fix_split_first_letter(cells[2]))
                        postal, phone, fax, email, website = parse_contact_cell(cells[3])
                        append_row(reg,
                                   name,
                                   location,
                                   postal,
                                   extract_city(postal) or extract_city(location),
                                   extract_zip(postal),
                                   phone,
                                   fax,
                                   email,
                                   website,
                                   cells[4])

    sqldict = bourange_same_length_array(sqldict)
    os.remove(pdf_path)
    rows_after = len(sqldict['Name'])
    print(f"[INFO] : {reg} collected {rows_after - rows_before} rows")


In [ ]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)

df = pd.DataFrame(sqldict)

df = df[df['Name'] != '']

outfile = os.path.join(scriptfolder, filename)
df.to_excel(outfile, sheet_name='SQL Ready', index=False)

# print(f"[INFO] : saved {len(df)} rows -> {outfile}")
# print(df['ListCode'].value_counts().sort_index())
